# SDC on a GPU — is there even an incumbent to beat?

The companion notebook (`ssj_gpu_colab.ipynb`) races **symmetric** eigensolvers
against cuSOLVER's `syevd` and loses. That result was clean and the reason was
specific: `syevd` costs only **5.4 gemm-equivalents** at n=2048 on a T4 and
falls toward its flop-ratio floor as n grows, so nothing built out of gemms
fits underneath it.

The **nonsymmetric** problem is a different contest, for two reasons that have
nothing to do with any algorithm:

1. On CPU, `dgeev` costs **131–181 gemm-equivalents** against `dsyevd`'s
   17–25. The incumbent is roughly 7× weaker in exactly the unit that decided
   the symmetric race. Spectral divide and conquer has *op-count parity* with
   it — 88 gemm-equivalents against 89, measured at n=400.
2. I expected cuSOLVER to provide **no general nonsymmetric eigensolver at
   all**, which would have left only a host round trip as the baseline. Cell 3
   tests that rather than assuming it, and on CuPy 14 **`cupy.linalg.eig`
   exists** and beats the round trip by 1.8×–2.0× at n ≥ 512. So there IS an
   incumbent; it is simply a much weaker one than `syevd` was on the symmetric
   side. The cell still reports the round trip, because it bounds what you pay
   if you keep the problem on the CPU.

SDC also has a property none of the symmetric methods here do: every
transformation it applies is an **orthogonal similarity**, so it has no basin
condition whatsoever. IPT, SSJ and the shears each stall or diverge on
non-normal input; SDC does not care.

**What this notebook decides.** Cell 1 measures the operations SDC actually
spends — gemm, LU+inverse, QR — in gemm-equivalents, because those numbers
predict its cost before any solver runs. Cell 3 then races two leaf strategies
whose CPU verdict this substrate is expected to invert.

Everything is self-contained; the repository is private, so there is nothing
to clone.


In [ ]:
# =====================================================================
#  1 - The probe: what do SDC's OWN operations cost on this card?
# =====================================================================
# The symmetric notebook probes the fp32:fp64 ratio, because mixed precision
# is what that family trades on. SDC trades on something else entirely: its
# far-field step is one gemm plus one matrix INVERSE, and its split needs one
# QR. So the numbers that predict SDC's cost here are those three, expressed
# in gemm-equivalents.
#
# For scale, the CPU measurements this port came from (N=1000, one gemm = 1):
#
#     gemm 1.00 | inverse 5.35 | QR 7.74 | dgeev 93.9 | dgees 88.7
#
# That 5.35 is why SDC is viable at all on CPU: dgeev is so far above gemm
# rate that a method doing several times more arithmetic still competes. If
# the inverse is proportionally CHEAPER here, SDC gets better; if the host
# round trip is cheap, SDC has nothing to beat. Both are measured below.

import shutil, subprocess, time
import numpy as np

if shutil.which("nvidia-smi") is None:
    raise SystemExit(
        "No GPU on this runtime (nvidia-smi is not installed).\n"
        "Runtime > Change runtime type > Hardware accelerator: GPU, "
        "then run this cell again.")
smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,compute_cap",
     "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
if not smi:
    raise SystemExit("nvidia-smi reports no GPU. Switch the runtime to GPU.")
print("GPU:", smi)

try:
    import cupy as cp
except Exception as e:
    raise SystemExit(f"cupy is unavailable on this runtime: {e}")
print("cupy", cp.__version__, "| numpy", np.__version__)


def _sync():
    cp.cuda.Device().synchronize()


def _best(fn, reps=5):
    fn(); _sync()
    b = float("inf")
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); _sync()
        b = min(b, time.perf_counter() - t0)
    return b


print(f"\n  {'n':>6}{'gemm ms':>10}{'inverse':>10}{'slogdet':>10}{'QR':>10}"
      f"{'  <- all in gemm-equivalents'}")
OPS = {}
for n in (512, 1024, 2048):
    rng = cp.random.default_rng(0)
    A = rng.standard_normal((n, n), dtype=cp.float64)
    A = A + n * cp.eye(n)                     # keep it comfortably invertible
    tg = _best(lambda: A @ A)
    ti = _best(lambda: cp.linalg.inv(A), reps=3)
    td = _best(lambda: cp.linalg.slogdet(A), reps=3)
    tq = _best(lambda: cp.linalg.qr(A), reps=3)
    OPS[n] = dict(gemm=tg, inv=ti / tg, slogdet=td / tg, qr=tq / tg)
    print(f"  {n:6d}{tg*1e3:10.2f}{ti/tg:10.2f}{td/tg:10.2f}{tq/tg:10.2f}")

print("""
  Reading this. A far-field Newton step costs one gemm plus one inverse (and
  one slogdet, if determinantal scaling is kept -- cell 2 makes that a
  measured choice rather than an assumption). So the per-step cost in
  gemm-equivalents is roughly 1 + inverse + slogdet, and SDC needs 12-20 of
  them. Multiply it out before running anything: if that product already
  exceeds what cell 3 reports for the host round trip, SDC cannot win here and
  the rest of the notebook is confirming a foregone conclusion.""")


In [ ]:
# =====================================================================
#  2 - SDC, backend-agnostic (same code path on numpy and cupy arrays)
# =====================================================================
# Ported from ssj.sdc (SSJ_LOG #24-31) and validated on the NumPy path against
# that implementation across Ginibre, planted-real, near-symmetric and
# companion matrices.
#
# Three findings from the CPU campaign are built in, each of which cost a tick
# to learn:
#
#   #31  THE NEWTON->NEWTON-SCHULZ HANDOFF MUST BE TESTED IN THE RIGHT NORM.
#        NS converges only inside ||I - X^2||_2 < 1. The iteration's natural
#        progress measure is ||I - X^2||_F/sqrt(n), an RMS quantity that sits
#        far BELOW the operator norm, so a fixed threshold on it tests the
#        wrong thing -- and how wrong depends on the spectrum. At dev < 0.9
#        (swept on Ginibre alone) a SYMMETRIC matrix entered NS outside its
#        region and never converged: 2712 ms against dgeev's 48 ms at n=400,
#        13448 ms at n=800. Since ||M||_2 <= ||M||_F, gating on
#        ||I - X^2||_F < 1 is guaranteed safe -- a 1/sqrt(n) SCALING LAW.
#
#   #30  THE FAR-FIELD CONVERGENCE GEMM IS OPTIONAL. A Newton step is X^2
#        (2n^3) plus the inverse, and half of it is a gemm whose only job is
#        the convergence test. But Delta = (X^-1 - X)/2 = X^-1(I - X^2)/2, so
#        the update norm is the same signal for O(n^2). Measured: Delta tracks
#        dev/2 to two digits through the endgame and is never small while dev
#        is large, so gating on it reproduces the same handoff while forming
#        X^2 once or twice instead of 8-11 times.
#
#   #28  THE LEAF IS 3n/5, NOT n/2. The centred split returns r = trace(P),
#        which lands near n/2 but essentially never ON it, so a leaf of n/2
#        sends one half back a few rows too big and buys a whole second
#        full-size sign iteration. Worth 1.10x-1.16x.

import time
import numpy as np


def _am(A):
    if type(A).__module__.partition(".")[0] == "cupy":
        import cupy
        return cupy
    return np


_G_CACHE = {}


def _inv_and_logdet(X, scaling):
    """Inverse, and log|det| only if the scaling actually needs it.

    THIS IS A MEASURED CHOICE, not an obvious one. Determinantal scaling
    mu = |det X|^(-1/n) needs log|det|, which costs a SECOND factorization
    unless it is shared -- and sharing it via lu_solve against a full identity
    (2n^3) is no cheaper in flops than getrf+getri (2n^3) plus a getrf for the
    determinant (2n^3/3). So on GPU the honest options are 2.67n^3 with
    scaling or 2n^3 without.

    Is the scaling worth 33%? Measured on CPU, iteration counts with
    determinantal scaling against unscaled Newton:

        Ginibre  n=200/400/800:  16/14/17  vs  17/15/19
        near-sym n=200/400/800:  23/22/25  vs  23/22/25

    So unscaled costs 0-13% more ITERATIONS to save 33% per far-field step.

    RACED ON A GPU (SSJ_LOG #36) AND UNSCALED WINS EVERYWHERE, so it is now
    the default: 2.14x/1.11x/1.26x faster at n=256/512/1024 with leaf=host,
    and 1.25x/1.51x/1.55x with leaf=deep, at unchanged accuracy. The n=256
    gain far exceeds the 33% the flop count predicts, which points at the
    other cost of the scaled branch: reading log|det| back to the host is a
    SYNC per Newton step, and at small n that dominates the arithmetic.
    """
    xp = _am(X)
    n = X.shape[0]
    if scaling == "none":
        return xp.linalg.inv(X), 0.0
    sgn, lad = xp.linalg.slogdet(X)
    return xp.linalg.inv(X), float(lad)


def sign_iterate(X, ns_frob=1.0, tol=1e-12, max_iter=60, scaling="none"):
    """Matrix sign: scaled Newton far away, Newton-Schulz once close.

        Newton         X <- (mu X + mu^-1 X^-1)/2,  mu = |det X|^(-1/n)
        Newton-Schulz  X <- X(3I - X^2)/2                      (2 gemms)

    Returns (S, iters, ok, n_newton, n_ns).
    """
    xp = _am(X)
    n = X.shape[0]
    sqn = np.sqrt(n)
    thresh = ns_frob / sqn                       # #31: a scaling law
    eye = xp.eye(n, dtype=X.dtype)

    nrm = float(xp.linalg.norm(X, ord="fro")) / sqn
    if nrm == 0.0:
        return X, 0, False, 0, 0
    X = X / nrm

    delta_prev = np.inf
    since_check = 0
    n_newton = n_ns = 0
    for it in range(1, max_iter + 1):
        if delta_prev < thresh or since_check >= 8:   # #30: gate the gemm
            since_check = 0
            X2 = X @ X
            dev = float(xp.linalg.norm(X2 - eye, ord="fro")) / sqn
            if not np.isfinite(dev):
                return X, it, False, n_newton, n_ns
            if dev < tol:
                return X, it, True, n_newton, n_ns
            if dev < thresh:
                X = X @ (1.5 * eye - 0.5 * X2)
                n_ns += 1
                delta_prev = 0.0
                continue
        else:
            since_check += 1
        Xi, lad = _inv_and_logdet(X, scaling)
        if scaling != "none" and not np.isfinite(lad):
            return X, it, False, n_newton, n_ns      # singular iterate
        mu = 1.0 if scaling == "none" else np.exp(-lad / n)
        Xn = 0.5 * (mu * X + Xi / mu)
        delta_prev = float(xp.linalg.norm(Xn - X, ord="fro")) / sqn
        X = Xn
        n_newton += 1
        if not np.isfinite(delta_prev):
            return X, it, False, n_newton, n_ns
    return X, max_iter, False, n_newton, n_ns


def _split_once(A, shift, ns_frob=1.0, tol=1e-12, scaling="none", st=None):
    """One spectral split at Re(z) = shift. Returns (B, r) or (None, code)."""
    xp = _am(A)
    n = A.shape[0]
    S, its, ok, nn_, ns_ = sign_iterate(A - shift * xp.eye(n, dtype=A.dtype),
                                        ns_frob=ns_frob, tol=tol,
                                        scaling=scaling)
    if st is not None:
        st["newton"] += nn_; st["ns"] += ns_; st["sign_calls"] += 1
    if not ok:
        return None, -2
    P = 0.5 * (xp.eye(n, dtype=A.dtype) + S)
    r = int(np.rint(float(xp.trace(P))))
    if r <= 0 or r >= n:
        return None, -1

    # Randomized range-finder instead of pivoted QR (cupy's qr has no
    # pivoting). The FIRST r columns must come from P and the rest from I - P,
    # IN THAT ORDER: SSJ_LOG #24 records building the basis from a pivoted QR
    # of [P, I-P] instead, whose column reordering destroys the range
    # separation and reported a bogus ||A21|| = 2.6e-01 on a symmetric matrix.
    key = (n, xp.__name__)
    if key not in _G_CACHE:
        _G_CACHE[key] = xp.asarray(
            np.random.default_rng(0x5D1).standard_normal((n, n)))
    G = _G_CACHE[key].astype(A.dtype, copy=False)
    Y = xp.empty((n, n), dtype=A.dtype)
    Y[:, :r] = P @ G[:, :r]
    Y[:, r:] = G[:, r:] - P @ G[:, r:]
    Q = xp.linalg.qr(Y)[0]
    B = Q.T @ (A @ Q)

    # The (2,1) block is zero in exact arithmetic; how far it misses IS the
    # split's backward error. For nonsymmetric A it scales with the OBLIQUE
    # projector norm ||P||, which equals 1 only in the symmetric case
    # (#24: ||P|| 3.2 -> 1.5e5 as cond(X) runs 10 -> 1e6). Hence the check.
    if float(xp.linalg.norm(B[r:, :r], ord="fro")) > \
            1e-6 * float(xp.linalg.norm(A, ord="fro")):
        return None, -3
    return B, r


def _leaf_closed_form(M):
    """1x1 or 2x2 in closed form, on the device. A conjugate pair shares a
    real part, so a vertical cut never separates one -- 2x2 always suffices."""
    xp = _am(M)
    if M.shape[0] == 1:
        return xp.asarray(M[0, 0], dtype=np.complex128).reshape(1)
    a, b, c, d = M[0, 0], M[0, 1], M[1, 0], M[1, 1]
    tr, det = a + d, a * d - b * c
    root = xp.sqrt(xp.asarray(tr * tr / 4.0 - det, dtype=np.complex128))
    return xp.stack([tr / 2.0 + root, tr / 2.0 - root])


def _to_host(A):
    return A if _am(A) is np else A.get()


# Leaf size at or above which the DEVICE solver is used by leaf_solver="auto".
# Measured directly (SSJ_LOG #38), and it is a real crossover, not a taste:
#
#     leaf size    host round trip    cupy.linalg.eig    winner
#           128            15.1 ms            54.1 ms    host
#           256            70.1 ms            94.2 ms    host
#           384           154.8 ms           166.6 ms    host
#           512           433.3 ms           246.2 ms    device
#           768           830.6 ms           455.7 ms    device
#
# So host wins THROUGH 384 and device takes over by 512; 450 sits between the
# measured bracket. An earlier value of 384 was too low -- host still won
# there. The shipped leaf of 3n/5 puts leaves at ~n/2, so for n in 512..1024
# this constant is what decides which solver the leaves actually use.
#
# Note how weakly cupy.linalg.eig scales: 128 -> 768 is 6x in size but only
# 8.4x in time, nothing like the 216x a cubic would give. It is dominated by
# fixed cost at these sizes, which is why it LOSES to a host round trip below
# ~450 and wins comfortably above it. Cell 3 re-measures the crossover rather
# than trusting this number on a different card.
LEAF_DEVICE_MIN = 450


def _leaf_solve(A, leaf_solver):
    """Eigenvalues of a leaf block. Three strategies, and which wins depends
    on the block SIZE, not on taste (see LEAF_DEVICE_MIN)."""
    xp = _am(A)
    n = A.shape[0]
    if n <= 2:
        return _leaf_closed_form(A)
    if xp is np:
        return np.asarray(np.linalg.eigvals(A), dtype=np.complex128)
    want_device = (leaf_solver == "device" or
                   (leaf_solver == "auto" and n >= LEAF_DEVICE_MIN))
    if want_device:
        try:
            return xp.asarray(xp.linalg.eig(A)[0], dtype=np.complex128)
        except Exception:
            pass          # no device eig on this CuPy: fall through to host
    return xp.asarray(np.linalg.eigvals(_to_host(A)), dtype=np.complex128)


def _now(xp):
    """Wall clock with a device sync -- ONLY called when profiling, because
    a sync per phase would perturb the very thing the race measures."""
    if xp is not np:
        xp.cuda.Device().synchronize()
    return time.perf_counter()


def sdc_eigvals(A, min_block=None, leaf_solver="host", ns_frob=1.0,
                tol=1e-12, scaling="none", profile=False, _depth=0, _st=None):
    """Eigenvalues of a general real matrix by spectral divide and conquer.

    leaf_solver : how leaf blocks are solved.
        "device" -- cupy.linalg.eig on the device, no transfer. SDC then acts
                    as a PRECONDITIONER for the vendor solver: split once,
                    then two half-size device solves.
        "host"   -- copy each leaf to the CPU for numpy.linalg.eigvals.
        "auto"   -- device at or above LEAF_DEVICE_MIN, host below. Neither
                    wins everywhere: at a leaf of 256 the host was 70.4 ms
                    against the device's 86.9 ms, and at 512 the host was
                    458.6 ms against 257.0 ms (SSJ_LOG #36 run).
        "deep"   -- recurse to 2x2 and never leave the device. Measured
                    4.4x-16.4x SLOWER on GPU; kept as a control only.
    """
    xp = _am(A)
    n = A.shape[0]
    if _st is None:
        _st = {"splits": 0, "sign_calls": 0, "newton": 0, "ns": 0,
               "leaves": 0, "fallbacks": 0,
               "t_split": 0.0, "t_leaf": 0.0}
    if min_block is None:
        min_block = 2 if leaf_solver == "deep" else max(2, 3 * n // 5)  # #28

    if n <= min_block or _depth >= 64:
        _st["leaves"] += 1
        t0 = _now(xp) if profile else 0.0
        out = _leaf_closed_form(A) if leaf_solver == "deep" and n <= 2 \
            else _leaf_solve(A, leaf_solver)
        if profile:
            _st["t_leaf"] += _now(xp) - t0
        return out, _st

    centre = float(xp.trace(A)) / n
    spread = float(xp.linalg.norm(A, ord="fro")) / np.sqrt(n)
    rng = np.random.default_rng(0xC0FFEE + _depth)
    B, r = None, -1
    t0 = _now(xp) if profile else 0.0
    for attempt in range(12):
        shift = centre if attempt == 0 else centre + spread * float(
            rng.standard_normal()) * 0.5 ** (attempt // 4)
        B, r = _split_once(A, shift, ns_frob, tol, scaling, _st)
        if B is not None:
            break
    if profile:
        _st["t_split"] += _now(xp) - t0
    if B is None:
        _st["fallbacks"] += 1
        return xp.asarray(np.linalg.eigvals(_to_host(A)),
                          dtype=np.complex128), _st

    _st["splits"] += 1
    w1, _ = sdc_eigvals(B[:r, :r], min_block, leaf_solver, ns_frob, tol,
                        scaling, profile, _depth + 1, _st)
    w2, _ = sdc_eigvals(B[r:, r:], min_block, leaf_solver, ns_frob, tol,
                        scaling, profile, _depth + 1, _st)
    return xp.concatenate([w1, w2]), _st


print("SDC loaded.")


In [ ]:
# =====================================================================
#  3 - Is there an incumbent? Then the race.
# =====================================================================
# Accuracy is asserted against dgeev before any time is believed: a routine
# that fails fast looks fast.

import time


def sync():
    cp.cuda.Device().synchronize()


def timed(fn, reps=3):
    fn(); sync()
    b = float("inf")
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); sync()
        b = min(b, time.perf_counter() - t0)
    return b


def matched_err(w, v, nrm):
    w = np.asarray(w.get() if hasattr(w, "get") else w, dtype=complex)
    v = list(np.asarray(v, dtype=complex))
    tot = 0.0
    for x in w:
        d = np.abs(x - np.array(v)); i = int(np.argmin(d))
        tot = max(tot, float(d[i])); v.pop(i)
    return tot / nrm


print("=== does this GPU have a general nonsymmetric eigensolver at all?")
HAVE_GPU_EIG = False
try:
    cp.linalg.eig(cp.asarray(np.random.default_rng(0).standard_normal((64, 64))))
    HAVE_GPU_EIG = True
    print("  cupy.linalg.eig EXISTS -- it becomes the incumbent below")
except Exception as e:
    print(f"  no general eig on the device: {type(e).__name__}: {e}")
    print("  -> the baseline is a HOST ROUND TRIP, transfers included.")
    print("     You cannot get eigenvalues of a device matrix without either")
    print("     a device solver or a copy, so excluding the copy would be")
    print("     measuring a solver that does not exist.")


def host_roundtrip(A):
    return cp.asarray(np.linalg.eigvals(A.get()))


SIZES = [256, 512, 1024]
print(f"\n  {'n':>6}{'method':>26}{'ms':>10}{'vs baseline':>13}{'gemm-eq':>9}"
      f"{'N/NS':>10}{'splits':>8}{'dlam':>10}")
rows = []
for n in SIZES:
    A_h = np.random.default_rng(2).standard_normal((n, n)) / np.sqrt(n)
    A = cp.asarray(A_h)
    nrm = float(np.linalg.norm(A_h, 2))
    wref = np.linalg.eigvals(A_h)

    t_gemm = timed(lambda: A @ A, reps=5)
    t_base = timed(lambda: host_roundtrip(A), reps=3)
    print(f"\n  n={n}: fp64 gemm {t_gemm*1e3:.2f} ms | baseline "
          f"{t_base*1e3:.1f} ms = {t_base/t_gemm:.1f} gemm-equivalents"
          f"   <-- SDC needs ~88, so this line decides the race")

    cands = [("baseline (host dgeev)", lambda: host_roundtrip(A))]
    if HAVE_GPU_EIG:
        cands.append(("cupy.linalg.eig", lambda: cp.linalg.eig(A)[0]))
    # leaf=deep measured 4.4x-16.4x SLOWER than leaf=host on a GPU
    # (SSJ_LOG #36) -- kept only as a control, not as a candidate
    for leaf, sc in (("device", "none"), ("auto", "none"),
                     ("host", "none"), ("deep", "none")):
        cands.append((f"SDC leaf={leaf} scale={sc}",
                      (lambda l=leaf, s=sc:
                       sdc_eigvals(A, leaf_solver=l, scaling=s))))

    for label, fn in cands:
        try:
            out = fn()
            w, st = out if isinstance(out, tuple) else (out, {})
            sync()
            e = matched_err(w, wref, nrm)
            if not (e < 1e-8):
                print(f"  {n:6d}{label:>27}{'':>10}{'':>13}{'':>9}{'':>10}"
                      f"{st.get('splits', 0):8}{e:10.1e}  NOT ACCURATE")
                continue
            t = timed(fn, reps=2)
            mark = ("   <-- BEATS the baseline"
                    if (t < t_base and not label.startswith("baseline")) else "")
            ns_ = f"{st.get('newton', 0)}/{st.get('ns', 0)}" if st else "--"
            print(f"  {n:6d}{label:>27}{t*1e3:10.1f}{t_base/t:12.2f}x"
                  f"{t/t_gemm:9.1f}{ns_:>10}{st.get('splits', 0):8}{e:10.1e}{mark}")
            rows.append((n, label, t, t_base, t_base / t, e))
        except Exception as ex:
            print(f"  {n:6d}{label:>27}  FAILED: {type(ex).__name__}: {ex}")

# ------------------------------------------------------------------------
# Where does the leaf solver actually cross over? One direct measurement,
# because the whole leaf policy turns on it and it is cheap to get.
print("\n=== leaf solver crossover: host round trip vs device eig, by block size")
print(f"  {'m':>6}{'host ms':>12}{'device ms':>12}{'winner':>10}")
for m in (128, 256, 384, 512, 768):
    Am = cp.asarray(np.random.default_rng(3).standard_normal((m, m)) / np.sqrt(m))
    th = timed(lambda: cp.asarray(np.linalg.eigvals(Am.get())), reps=3)
    try:
        td = timed(lambda: cp.linalg.eig(Am)[0], reps=3)
        win = "host" if th < td else "device"
        print(f"  {m:6d}{th*1e3:12.1f}{td*1e3:12.1f}{win:>10}")
    except Exception as e:
        print(f"  {m:6d}{th*1e3:12.1f}{'--':>12}{'host':>10}  ({type(e).__name__})")
print("  -> set LEAF_DEVICE_MIN in cell 2 to the size where device takes over")

print("""
  Reading this table

  THE ROW THAT MATTERS is cupy.linalg.eig, not the host baseline -- where it
  exists it is the real incumbent, and on the run behind SSJ_LOG #36 it beat
  the host round trip by 1.8x-2.0x at n>=512. Compare SDC against THAT.

  THE gemm-eq COLUMN IS MIXED-DEVICE HERE, so do not read it as the campaign's
  unit. The host baseline's numerator is CPU dgeev while the denominator is
  GPU gemm, and leaf=host puts a CPU solve inside SDC's own numerator too. The
  campaign's "SDC needs ~88" was measured CPU-over-CPU. Use the millisecond
  and ratio columns; the gemm-eq column is a rough scale only.

  leaf=device makes SDC a PRECONDITIONER for the vendor solver rather than a
  replacement for it: one split on the device, then two half-size device
  eigs. That wins whenever the split costs less than the difference between
  one full solve and two half solves -- which is not a given, because
  cupy.linalg.eig measured only 3.07x between n=512 and n=1024, not the 8x
  a cubic would give, so halving n saves far less than it looks.

  leaf=auto exists because neither leaf solver wins everywhere: the crossover
  measured above sits between 256 and 512, and the shipped leaf of 3n/5 puts
  leaves at ~n/2 -- right on top of it.

  leaf=deep is a CONTROL, not a candidate. SSJ_LOG #25 measured deep recursion
  4x slower on CPU, and the expectation here was that a device without geev
  would flip that. It did not -- #36 measured deep 4.4x-16.4x SLOWER on GPU,
  because the premise was wrong twice over: cupy.linalg.eig exists, and the
  bottom levels issue hundreds of tiny kernels (526 splits at n=1024).

  scale=none is now the default and wins everywhere, by more than its 33%
  flop saving predicts -- the scaled branch also syncs log|det| to the host
  every Newton step.""")


## What the first run measured (SSJ_LOG #36)

**SDC beats the real incumbent at n=512 and loses at n=1024.** Against
`cupy.linalg.eig`, with `leaf=host, scale=none`:

| n | cupy.linalg.eig | SDC | |
|---|---|---|---|
| 256 | 86.9 ms | 80.3 ms | **1.08×** |
| 512 | 257.0 ms | 226.5 ms | **1.13×** |
| 1024 | 790.3 ms | 1271.5 ms | 0.62× |

That is the campaign's **first outright win over a vendor eigensolver on any
substrate** — narrow, size-limited, and on the nonsymmetric side exactly where
SSJ_LOG #24 predicted the opening would be.

**Accuracy degrades with n and needs watching**: 3.8e-14 → 2.9e-13 → 3.4e-11
against `cupy.linalg.eig`'s 5.2e-15 → 9.2e-15 → 1.3e-14. Still inside the bar,
but the trend is the wrong way and it is not yet explained.

**Watch for a fallback.** The `splits` column and any `NOT ACCURATE` row tell
you whether SDC did its work or quietly fell back to `eigvals`. A fallback
that returns the right answer fast is measuring LAPACK, not SDC.

## What this notebook does not settle

* **fp32.** The whole sign iteration could run in single precision with a
  final refinement, and on a card where fp32 is many times fp64 that is the
  obvious next lever. It is untested here because the refinement ladder's
  behaviour on *non-symmetric* input is itself unmeasured — SSJ_LOG #24 found
  the ladder loses its Newton–Schulz half there (non-symmetric eigenvectors
  are not orthonormal, so re-orthonormalising would destroy the answer), and
  only the consult-A half survives.
* **Batched small blocks.** `leaf=deep` recurses to 2×2 and issues one kernel
  per block at the bottom levels. Level *k* holds 2^k blocks of the same size,
  so they could be batched — which is what made SSJ-BC viable on GPU in the
  companion notebook. Untested here, and it is the obvious fix if `deep` loses
  on launch overhead rather than on arithmetic.
* **Eigenvectors.** This computes eigenvalues only, as `ssj.sdc` does. The
  orthogonal similarities are accumulated implicitly and thrown away; keeping
  them costs one more gemm per split.

## Provenance

Ported from `ssj.sdc` and validated on the NumPy path against that
implementation across Ginibre, planted-real, near-symmetric and companion
matrices at n = 128 and 256 — both leaf strategies land 7.9e-15 to 1.0e-11,
against the reference's 8.7e-15 to 1.9e-13.

Three CPU findings are built in and commented at the point of use: the
Frobenius handoff bound (#31 — a fixed threshold in the wrong norm made
symmetric input 54× slower than `dgeev`), the far-field gate (#30 — the
convergence-test gemm is skipped while the update norm says we are far), and
the 3n/5 leaf (#28 — n/2 buys a whole second sign iteration).
